### Ensemble Methods — Phase 4 & 5: Blending + Full Comparison + Regression Ensemble
### Telco Churn + Ames Housing Datasets
 
Steps:
  4. Manual blending on Telco Churn
  5. Full comparison table — all techniques
  6. StackingRegressor on Ames Housing (shows ensembles work for regression)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                               GradientBoostingRegressor, StackingRegressor,
                               VotingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, mean_squared_error,
                             mean_absolute_error)
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
import joblib
import warnings
warnings.filterwarnings("ignore")
 
plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e5e5e5"})
 
SEED = 42

### MANUAL BLENDING

In [ ]:
#    Blending vs stacking:
#      Stacking  — OOF predictions cover 100% of training data
#      Blending  — hold out a fixed 20% "blend set"; train base learners
#                  on remaining 80%; predict blend set → meta-train matrix
#    Blending is simpler and faster but wastes 20% of training data.
#    Stacking is preferred on small-medium datasets. Blending is used
#    on very large datasets where 5-fold CV is too slow.
# ════════════════════════════════════════════════════════════════════════════
 
print(f"\n── Manual Blending ────────────────────────────────────────")
 
# Split training data into base-train and blend-holdout
X_base_train, X_blend, y_base_train, y_blend = train_test_split(
    X_train, y_train, test_size=0.20,
    stratify=y_train, random_state=SEED
)
print(f"  Base train   : {len(X_base_train):,}")
print(f"  Blend holdout: {len(X_blend):,}  ← meta-learner trains here")
print(f"  Test set     : {len(X_test):,}   ← final evaluation")
 
# Train base learners on base_train only
pos_weight = (y_base_train == 0).sum() / (y_base_train == 1).sum()
 
blend_base_defs = {
    "Logistic Regression": Pipeline([
        ("pre",   joblib.loads(joblib.dumps(
            fitted_models["Logistic Regression"].named_steps["pre"]))),
        ("model", LogisticRegression(max_iter=1000,
                                      class_weight="balanced",
                                      random_state=SEED)),
    ]),
    "Random Forest": Pipeline([
        ("pre",   joblib.loads(joblib.dumps(
            fitted_models["Random Forest"].named_steps["pre"]))),
        ("model", RandomForestClassifier(n_estimators=200,
                                          class_weight="balanced",
                                          random_state=SEED, n_jobs=-1)),
    ]),
    "XGBoost": Pipeline([
        ("pre",   joblib.loads(joblib.dumps(
            fitted_models["XGBoost"].named_steps["pre"]))),
        ("model", XGBClassifier(n_estimators=300, learning_rate=0.05,
                                 max_depth=5, subsample=0.8,
                                 scale_pos_weight=pos_weight,
                                 random_state=SEED,
                                 eval_metric="logloss", verbosity=0)),
    ]),
    "LightGBM": Pipeline([
        ("pre",   joblib.loads(joblib.dumps(
            fitted_models["LightGBM"].named_steps["pre"]))),
        ("model", LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                  max_depth=5, class_weight="balanced",
                                  random_state=SEED, verbose=-1)),
    ]),
    "KNN": Pipeline([
        ("pre",   joblib.loads(joblib.dumps(
            fitted_models["KNN"].named_steps["pre"]))),
        ("model", KNeighborsClassifier(n_neighbors=11, n_jobs=-1)),
    ]),
}
 
# Fit on 80% base train
blend_fitted = {}
print(f"  Training base learners on 80% of training data...")
for name, pipe in blend_base_defs.items():
    pipe.fit(X_base_train, y_base_train)
    blend_fitted[name] = pipe
    blend_auc = roc_auc_score(
        y_blend, pipe.predict_proba(X_blend)[:, 1])
    print(f"    {name:<22} blend AUC={blend_auc:.4f}")
 
# Build blend feature matrix from holdout predictions
blend_train_matrix = np.column_stack([
    pipe.predict_proba(X_blend)[:, 1]
    for pipe in blend_fitted.values()
])
blend_test_matrix  = np.column_stack([
    pipe.predict_proba(X_test)[:, 1]
    for pipe in blend_fitted.values()
])
 
# Train meta-learner on blend holdout
blend_meta = LogisticRegression(max_iter=1000, random_state=SEED)
blend_meta.fit(blend_train_matrix, y_blend)
 
blend_prob = blend_meta.predict_proba(blend_test_matrix)[:, 1]
blend_pred = (blend_prob >= 0.5).astype(int)
blend_auc  = roc_auc_score(y_test, blend_prob)
blend_f1   = f1_score(y_test, blend_pred)
blend_rec  = recall_score(y_test, blend_pred)
blend_prec = precision_score(y_test, blend_pred, zero_division=0)
 
print(f"\n  Blending results:")
print(f"    AUC-ROC   : {blend_auc:.4f}  "
      f"({blend_auc - best_base_auc:+.4f} vs best single)")
print(f"    F1        : {blend_f1:.4f}")
print(f"\n  Stacking vs blending:")
print(f"    Stacking  : {sk_stack_auc:.4f}")
print(f"    Blending  : {blend_auc:.4f}")
print(f"    Difference: {sk_stack_auc - blend_auc:+.4f}  "
      f"(stacking uses more training data → usually wins)")

### FULL COMPARISON TABLE — ALL TECHNIQUES

In [ ]:
# Collect all results
all_entries = {}
 
# Base learners
for name, pipe in fitted_models.items():
    yp = pipe.predict_proba(X_test)[:, 1]
    ypred = pipe.predict(X_test)
    all_entries[name] = {
        "Type"     : "Base learner",
        "AUC-ROC"  : round(roc_auc_score(y_test, yp), 4),
        "F1"       : round(f1_score(y_test, ypred), 4),
        "Recall"   : round(recall_score(y_test, ypred), 4),
        "Precision": round(precision_score(y_test, ypred,
                                            zero_division=0), 4),
    }
 
# Voting (reconstruct from Phase 2 for completeness)
soft_voter = VotingClassifier(
    estimators=[(n, p) for n, p in fitted_models.items()],
    voting="soft", n_jobs=-1
)
soft_voter.fit(X_train, y_train)
sv_prob  = soft_voter.predict_proba(X_test)[:, 1]
sv_pred  = soft_voter.predict(X_test)
all_entries["Soft Voting"] = {
    "Type"     : "Voting",
    "AUC-ROC"  : round(roc_auc_score(y_test, sv_prob), 4),
    "F1"       : round(f1_score(y_test, sv_pred), 4),
    "Recall"   : round(recall_score(y_test, sv_pred), 4),
    "Precision": round(precision_score(y_test, sv_pred, zero_division=0), 4),
}
 
# Stacking
all_entries["Stacking (sklearn)"] = {
    "Type"     : "Stacking",
    "AUC-ROC"  : round(sk_stack_auc, 4),
    "F1"       : round(f1_score(y_test,
                                 (stack_preds["sklearn_stack_prob"] >= 0.5)
                                 .astype(int)), 4),
    "Recall"   : round(recall_score(y_test,
                                     (stack_preds["sklearn_stack_prob"] >= 0.5)
                                     .astype(int)), 4),
    "Precision": round(precision_score(y_test,
                                        (stack_preds["sklearn_stack_prob"] >= 0.5)
                                        .astype(int), zero_division=0), 4),
}
 
# Blending
all_entries["Blending"] = {
    "Type"     : "Blending",
    "AUC-ROC"  : round(blend_auc, 4),
    "F1"       : round(blend_f1, 4),
    "Recall"   : round(blend_rec, 4),
    "Precision": round(blend_prec, 4),
}
 
comp_df = (pd.DataFrame(all_entries).T
             .sort_values("AUC-ROC", ascending=False))
comp_df["vs Best Single"] = (comp_df["AUC-ROC"].astype(float)
                              - best_base_auc).round(4)
 
print(f"\n{'='*70}")
print("FULL COMPARISON TABLE — ALL ENSEMBLE TECHNIQUES")
print(f"{'='*70}")
print(comp_df.to_string())
 
best_overall     = comp_df.index[0]
best_overall_auc = comp_df.iloc[0]["AUC-ROC"]
print(f"\n✓ Best overall: {best_overall}  AUC={best_overall_auc:.4f}")
print(f"  Improvement over best single model: "
      f"{best_overall_auc - best_base_auc:+.4f}")
 
 
# ── Full comparison chart ─────────────────────────────────────────────────
type_colors = {"Base learner": "#4C72B0", "Voting": "#FF7F0E",
               "Stacking": "#E07070", "Blending": "#9467BD"}
 
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Full Ensemble Comparison — Telco Churn",
             fontweight="bold", fontsize=13)
 
names_all  = comp_df.index.tolist()
aucs_all   = comp_df["AUC-ROC"].astype(float).tolist()
types_all  = comp_df["Type"].tolist()
colors_all = [type_colors[t] for t in types_all]
 
ax = axes[0]
bars = ax.bar(names_all, aucs_all, color=colors_all,
              width=0.55, edgecolor="white")
ax.set_ylim(min(aucs_all) - 0.01, max(aucs_all) + 0.015)
ax.set_ylabel("Test AUC-ROC")
ax.set_title("AUC-ROC — all techniques")
ax.tick_params(axis="x", rotation=30)
for bar, v in zip(bars, aucs_all):
    ax.text(bar.get_x()+bar.get_width()/2,
            bar.get_height()+0.0005,
            f"{v:.4f}", ha="center", fontsize=8)
legend_patches = [mpatches.Patch(color=c, label=t)
                  for t, c in type_colors.items()]
ax.legend(handles=legend_patches, fontsize=8, loc="lower right")
 
# Gain chart
ax2 = axes[1]
gains = comp_df["vs Best Single"].astype(float).tolist()
gain_cols = ["#2ca02c" if g > 0 else
             "#aaaaaa" if g == 0 else "#E07070"
             for g in gains]
ax2.bar(names_all, gains, color=gain_cols,
        width=0.55, edgecolor="white")
ax2.axhline(0, color="#333", linewidth=0.8)
ax2.set_ylabel("AUC gain vs best single model")
ax2.set_title("Improvement over best single model")
ax2.tick_params(axis="x", rotation=30)
for i, g in enumerate(gains):
    ax2.text(i, g + (0.0003 if g >= 0 else -0.0008),
             f"{g:+.4f}", ha="center", fontsize=8)
 
plt.tight_layout()
plt.savefig("ensemble_phase4_full_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

### STACKING FOR REGRESSION — AMES HOUSING

In [ ]:
#    Demonstrates ensembles work equally well on regression tasks.
#    Reuse the Ames Housing dataset from Project 1.
# ════════════════════════════════════════════════════════════════════════════
 
print(f"\n{'='*65}")
print("REGRESSION ENSEMBLE — Ames Housing")
print(f"{'='*65}")
 
try:
    ames = pd.read_csv("train.csv")
    print(f"Ames Housing loaded: {ames.shape}")
except FileNotFoundError:
    print("  train.csv not found — skipping regression ensemble.")
    print("  Download from: kaggle.com/competitions/house-prices-advanced-regression-techniques")
    ames = None
 
if ames is not None:
    # ── Quick preprocessing (mirrors Project 1) ───────────────────────────
    ames["LogSalePrice"] = np.log1p(ames["SalePrice"])
 
    # Fill structural NaNs
    none_cols = ["PoolQC","MiscFeature","Alley","Fence","FireplaceQu",
                 "GarageType","GarageFinish","GarageQual","GarageCond",
                 "BsmtQual","BsmtCond","BsmtExposure","BsmtFinType1",
                 "BsmtFinType2","MasVnrType"]
    zero_cols = ["GarageYrBlt","GarageArea","GarageCars","BsmtFinSF1",
                 "BsmtFinSF2","BsmtUnfSF","TotalBsmtSF",
                 "BsmtFullBath","BsmtHalfBath","MasVnrArea"]
    for c in none_cols:
        if c in ames.columns: ames[c] = ames[c].fillna("None")
    for c in zero_cols:
        if c in ames.columns: ames[c] = ames[c].fillna(0)
    ames["LotFrontage"] = ames.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median()))
 
    # New features
    ames["TotalSF"]  = ames["TotalBsmtSF"] + ames["1stFlrSF"] + ames["2ndFlrSF"]
    ames["HouseAge"] = ames["YrSold"] - ames["YearBuilt"]
 
    target_col = "LogSalePrice"
    drop_cols  = ["Id","SalePrice","LogSalePrice","YearBuilt",
                  "YearRemodAdd","YrSold","MoSold"]
    X_ames = ames.drop(columns=[c for c in drop_cols if c in ames.columns])
    y_ames = ames[target_col]
 
    num_ames = X_ames.select_dtypes(include=np.number).columns.tolist()
    cat_ames = X_ames.select_dtypes(include="object").columns.tolist()
 
    num_pipe_r = Pipeline([("imp", SimpleImputer(strategy="median")),
                            ("sc",  RobustScaler())])
    cat_pipe_r = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("enc", OneHotEncoder(handle_unknown="ignore",
                                                  sparse_output=False))])
    pre_r = ColumnTransformer([("num", num_pipe_r, num_ames),
                                ("cat", cat_pipe_r, cat_ames)])
 
    X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
        X_ames, y_ames, test_size=0.2, random_state=SEED
    )
 
    def rmse(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))
 
    def dollar_rmse(log_rmse):
        return np.expm1(log_rmse)
 
    # ── Base regressors ───────────────────────────────────────────────────
    reg_base = {
        "Ridge": Pipeline([("pre", pre_r),
            ("model", Ridge(alpha=10.0))]),
        "Lasso": Pipeline([("pre", pre_r),
            ("model", Lasso(alpha=0.001, max_iter=5000))]),
        "Random Forest": Pipeline([("pre", pre_r),
            ("model", RandomForestRegressor(n_estimators=200,
                                             random_state=SEED, n_jobs=-1))]),
        "XGBoost": Pipeline([("pre", pre_r),
            ("model", XGBRegressor(n_estimators=300, learning_rate=0.05,
                                    max_depth=4, subsample=0.8,
                                    random_state=SEED, verbosity=0))]),
        "LightGBM": Pipeline([("pre", pre_r),
            ("model", LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                     max_depth=4, random_state=SEED,
                                     verbose=-1))]),
    }
 
    print(f"\nTraining regression base learners...")
    reg_results = {}
    reg_fitted  = {}
 
    for name, pipe in reg_base.items():
        pipe.fit(X_tr_a, y_tr_a)
        reg_fitted[name] = pipe
        yp   = pipe.predict(X_te_a)
        rmse_log  = rmse(y_te_a, yp)
        rmse_dol  = dollar_rmse(rmse_log)
        r2    = 1 - np.sum((y_te_a - yp)**2) / np.sum((y_te_a - y_te_a.mean())**2)
        reg_results[name] = {
            "Type"     : "Base",
            "RMSE(log)": round(rmse_log, 5),
            "~$ RMSE"  : round(rmse_dol, 0),
            "R²"       : round(r2, 4),
        }
        print(f"  {name:<16} RMSE(log)={rmse_log:.4f}  "
              f"≈${rmse_dol:,.0f}  R²={r2:.4f}")
 
    # ── StackingRegressor ─────────────────────────────────────────────────
    print(f"\nTraining StackingRegressor...")
 
    stacking_reg = StackingRegressor(
        estimators      = [(n, p) for n, p in reg_fitted.items()],
        final_estimator = Ridge(alpha=1.0),
        cv              = KFold(n_splits=5, shuffle=True, random_state=SEED),
        passthrough     = False,
        n_jobs          = -1,
    )
    stacking_reg.fit(X_tr_a, y_tr_a)
    sr_pred    = stacking_reg.predict(X_te_a)
    sr_rmse    = rmse(y_te_a, sr_pred)
    sr_dol     = dollar_rmse(sr_rmse)
    sr_r2      = 1 - np.sum((y_te_a - sr_pred)**2) / \
                     np.sum((y_te_a - y_te_a.mean())**2)
 
    reg_results["Stacking (Ridge meta)"] = {
        "Type"     : "Stacking",
        "RMSE(log)": round(sr_rmse, 5),
        "~$ RMSE"  : round(sr_dol, 0),
        "R²"       : round(sr_r2, 4),
    }
    print(f"  {'Stacking':<16} RMSE(log)={sr_rmse:.4f}  "
          f"≈${sr_dol:,.0f}  R²={sr_r2:.4f}")
 
    reg_df = pd.DataFrame(reg_results).T.sort_values("RMSE(log)")
    best_single_reg = reg_df[reg_df["Type"] == "Base"].iloc[0]
 
    print(f"\n{'='*65}")
    print("REGRESSION ENSEMBLE COMPARISON")
    print(f"{'='*65}")
    print(reg_df.to_string())
    print(f"\n  Best single model : {reg_df[reg_df['Type']=='Base'].index[0]}  "
          f"RMSE(log)={best_single_reg['RMSE(log)']:.5f}")
    print(f"  Stacking          : RMSE(log)={sr_rmse:.5f}  "
          f"({sr_rmse - float(best_single_reg['RMSE(log)']):+.5f})")
    print(f"  Dollar improvement: "
          f"≈${float(best_single_reg['~$ RMSE']) - sr_dol:,.0f} lower error")
 
    # ── Regression comparison chart ───────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Regression Ensemble — Ames Housing",
                 fontweight="bold", fontsize=13)
 
    names_r  = reg_df.index.tolist()
    rmses_r  = reg_df["RMSE(log)"].astype(float).tolist()
    r2s_r    = reg_df["R²"].astype(float).tolist()
    types_r  = reg_df["Type"].tolist()
    cols_r   = ["#E07070" if t == "Stacking" else "#4C72B0"
                for t in types_r]
 
    ax = axes[0]
    bars = ax.bar(names_r, rmses_r, color=cols_r,
                  width=0.55, edgecolor="white")
    ax.set_ylim(min(rmses_r) - 0.002, max(rmses_r) + 0.005)
    ax.set_ylabel("RMSE (log scale)")
    ax.set_title("RMSE (log) — lower is better\n"
                 "(red = stacking ensemble)")
    ax.tick_params(axis="x", rotation=25)
    for bar, v in zip(bars, rmses_r):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.0002,
                f"{v:.4f}", ha="center", fontsize=8.5)
 
    ax2 = axes[1]
    bars2 = ax2.bar(names_r, r2s_r, color=cols_r,
                    width=0.55, edgecolor="white")
    ax2.set_ylim(min(r2s_r) - 0.01, 1.0)
    ax2.set_ylabel("R²")
    ax2.set_title("R² — higher is better")
    ax2.tick_params(axis="x", rotation=25)
    for bar, v in zip(bars2, r2s_r):
        ax2.text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+0.001,
                 f"{v:.4f}", ha="center", fontsize=8.5)
 
    plt.tight_layout()
    plt.savefig("ensemble_phase5_regression.png",
                dpi=150, bbox_inches="tight")
    plt.show()
 
    # Predicted vs actual for stacking
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(np.expm1(y_te_a), np.expm1(sr_pred),
               alpha=0.4, s=12, color="#E07070")
    lims = [0, max(np.expm1(y_te_a).max(), np.expm1(sr_pred).max())]
    ax.plot(lims, lims, "--", color="#333", linewidth=1.5,
            label="Perfect prediction")
    ax.set_xlabel("Actual Sale Price ($)")
    ax.set_ylabel("Predicted Sale Price ($)")
    ax.set_title(f"StackingRegressor — Predicted vs Actual\n"
                 f"RMSE ≈ ${sr_dol:,.0f}  R²={sr_r2:.4f}",
                 fontweight="bold")
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f"${x/1000:.0f}k"))
    ax.legend()
    plt.tight_layout()
    plt.savefig("ensemble_phase5_regression_scatter.png",
                dpi=150, bbox_inches="tight")
    plt.show()